### Test data for ML deployment.

Based on post_processing_deployment_version.ipynb script.
#### TODO: add canonicalId to data extract (and other Marceli fields)
#### TODO: remove chartTime rounding from raw data save?
#### TODO: add jitter to all datetimes (same shift for patient at each step in the pipeline).

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import timedelta
import os
os.chdir('../streamlit/')
from utilities import LocalDatabaseWrapper
os.chdir('../development')

In [2]:
project_name = "smartt_bmj_open_plus"
database_path = Path("../data")

In [3]:
local_db = LocalDatabaseWrapper(
            database_path / f"{project_name}.db"
        )

In [4]:
final_mapping = local_db.query_pd(
    "select * from final_mapping;"
    
)

In [5]:
for var in final_mapping.schemaVariable.unique():
    count = (final_mapping.schemaVariable==var).sum()
    final_mapping.loc[final_mapping.schemaVariable==var,'priority'] = range(count)

In [6]:
final_mapping['suggestedParameterName'] = [
    row.schemaVariable + '_' + str(row.priority) for ri,row in final_mapping.iterrows()
]

In [7]:
final_cohort = pd.read_json('..\data\processed\linked_cohort_181024.json', convert_dates=['admitStoreDateTime', 'rfd_datetime'])

In [9]:
dummy_ward = {
    eid: np.random.randint(100)
    for eid in final_cohort.encounterId.sample(5)
}

In [10]:
dummy_ward

{2972: 35, 28969: 9, 2638: 14, 10582: 98, 15425: 81}

In [11]:
orig_features = [
    'Glasgow Coma Scale', 'temperature', 'arterial blood pressure (mean)', 'Fio2',
    'arterial oxygen saturation', 'respiratory rate', 'Heart rate', 'actual bicarbonate',
    'Hemoglobin', 'platelet count', 'Sodium', 'potassium',
    'creatinine', 'urea', 'Bilirubin', 'positive end-expiratory pressure', 'hours_since_admission'
]
new_features = [
'Pao2',
 'C-reactive protein',
 'arterial blood pressure (systolic)',
 'arterial blood pressure (diastolic)',
 'noninvasive blood pressure (systolic)',
 'noninvasive blood pressure (diastolic)',
 'tidal volume',
 'peripheral oxygen saturation',
 'Richmond Agitation-Sedation Scale',
 'Pao2/Fio2',
 'Urine output'
]
all_features = orig_features + new_features

In [12]:
df_joined = None
for EID in dummy_ward.keys():
    df = local_db.query_pd(
        f"select * from full_extract where encounterId = {EID};"
    )
    df['chartTime'] = pd.to_datetime(df['chartTime'])
    min_hour = final_cohort[final_cohort.encounterId == EID].iloc[0].admitStoreDateTime.round('h')
    df['hours_since_admission'] = [(t - min_hour).total_seconds() / (60**2) for t in df.chartTime]
    df['admitStoreDateTime'] = min_hour
    df = df.merge(final_mapping, on=['interventionId', 'attributeId'])
    
    if df_joined is None:
        df_joined = df.copy()
    else:
        df_joined = pd.concat([df_joined, df])

In [13]:
def replace_eid(df, dummy_ward):
    df['encounterId'] = [
        dummy_ward[eid]
        for eid in df.encounterId
    ]
    return df

In [14]:
df_joined = replace_eid(df_joined, dummy_ward)

In [15]:
df_joined = df_joined[df_joined['schemaVariable'].isin(all_features)]

##### Does this preserve the original pipeline? (Previously used raw terseForm in below routines)

In [16]:
df_joined.loc[:,'value'] = pd.to_numeric(df_joined['terseForm'], errors='coerce')

In [17]:
keep_columns = [
    'attributeId', 'interventionId', 'encounterId',
    'value', 'hours_since_admission', 'schemaVariable',
    'admitStoreDateTime', 'unitOfMeasure',
    'chartTime', 'utcChartTime', 'priority'
]

In [18]:
df_joined = df_joined[keep_columns]

In [19]:
save_to_path = Path(f"E:/SMARTT/data/processed/test_deployment_data/{project_name}/raw_data/")

In [20]:
scaling = 20
noise_scale = df_joined.groupby('schemaVariable').agg({'value': 'mean'})
noise_scale.value /= scaling
noise_scale = noise_scale.to_dict()['value']

In [21]:
additive_noise = [
    row.value + np.random.normal(scale=np.abs(noise_scale[row.schemaVariable]), size=1)[0]
    for ri, row in df_joined.iterrows()
]

In [22]:
df_joined['value'] += additive_noise

In [23]:
df_joined.reset_index(inplace=True)
df_joined.to_json(save_to_path / 'dummy_ward_data.json', orient='index', indent=2)

#### We now save the grouped data for each dummy patient:

In [37]:
save_to_path = Path(f"E:/SMARTT/data/processed/test_deployment_data/{project_name}/first_records/")

In [40]:
for ei, original_EID in enumerate(dummy_ward.keys()):

    EID = dummy_ward[original_EID]
    
    df = df_joined[df_joined.encounterId==EID].copy()
    df_grouped = df.groupby(['hours_since_admission', 'schemaVariable']).agg('first')
    df_grouped.reset_index().to_json(save_to_path / f"{EID}_first_records.json", orient='index', indent=2)

### We now produce the complete time series for all dummy ward patients:

##### Note: we use forward fill here, but this still has some NaNs where the variables is missing completely or missing at the start of the series. We also include the raw variable columns so that alternative imputation methods can be used. 

We also add the outcome label at each time point according to:
- if encounter has negative outcome (0): label is 0 always
- if encounter has a discharge timestamp and positive outcome, label is 1 from timestamp onwards
- if encounter has no discharge timestamp and positive outcome, label is 1 at end.

In [24]:
COL = 'value'

In [25]:
save_to_path = Path(f"E:/SMARTT/data/processed/test_deployment_data/{project_name}/complete_encounter_series/")

In [30]:
encounter_series.columns

Index(['time', 'hours_since_admission', 'Glasgow Coma Scale', 'temperature',
       'arterial blood pressure (mean)', 'Fio2', 'arterial oxygen saturation',
       'respiratory rate', 'Heart rate', 'actual bicarbonate', 'Hemoglobin',
       'platelet count', 'Sodium', 'potassium', 'creatinine', 'urea',
       'Bilirubin', 'positive end-expiratory pressure',
       'hours_since_admission'],
      dtype='object')

In [35]:
rfd_na = 0
for ei, original_EID in enumerate(dummy_ward.keys()):

    EID = dummy_ward[original_EID]
    
    df = df_joined[df_joined.encounterId==EID].copy()
    df_grouped = df.groupby(['hours_since_admission', 'schemaVariable']).agg('first')

    max_time = df.chartTime.max().round('h')
    admit_time = final_cohort[final_cohort.encounterId == original_EID].iloc[0].admitStoreDateTime.round('h')
    rfd_time = final_cohort[final_cohort.encounterId == original_EID].iloc[0].rfd_datetime.round('h')
    outcome = final_cohort[final_cohort.encounterId == original_EID].iloc[0].outcome

    encounter_series = pd.DataFrame({
        'time': [
            admit_time + timedelta(hours=i)
            for i in range(int((max_time - admit_time).total_seconds() / 60**2))
        ],
        'hours_since_admission': [
            float(i)
            for i in range(int((max_time - admit_time).total_seconds() / 60**2))
        ]
    })

    keep_columns = ['time', 'hours_since_admission'] 
    for var in all_features:
        if var != 'hours_since_admission':
            encounter_series = pd.merge_asof(
                encounter_series, 
                df_grouped.reset_index()[df_grouped.reset_index()['schemaVariable']==var], 
                on='hours_since_admission',
                direction='backward',
                tolerance=1
            )[keep_columns + [COL]].rename(columns={COL:var})
            keep_columns.append(var)

            encounter_series[var + '_ffill'] = encounter_series[var].ffill().infer_objects(copy=False)

    # add outcome labels:
    if outcome == 0:
        encounter_series['label'] = [0 for i in range(len(encounter_series))]
    elif pd.isna(rfd_time):
        rfd_na+= 1
        encounter_series['label'] = [0 for i in range(len(encounter_series) - 1)] + [1]
    else:
        encounter_series['label'] = [
            t >= rfd_time.round('h')
            for t in encounter_series.time
        ]

    print(ei, EID, encounter_series['label'].sum(), encounter_series['label'].sum() / len(encounter_series))
    encounter_series.to_csv(save_to_path / f"{EID}_encounter_series.csv")
    if ei > 500:
        break

print(rfd_na)

0 35 18 0.5454545454545454
1 9 0 0.0
2 14 7 0.3333333333333333
3 98 0 0.0
4 81 11 0.2972972972972973
0


## COMPILED DUMMY DATA UP TO HERE.

# TODO: complete processing and feature engineering and save outputs.

In [779]:
final_cohort.drop_duplicates(subset='encounterId', keep='first', inplace=True)

In [847]:
test_cohort = final_cohort.sample(frac=0.2, random_state=42)

In [848]:
train_cohort = final_cohort[~final_cohort.index.isin(test_cohort.index)]

In [849]:
(~train_cohort['Declared clinically ready for discharge'].isna()).sum()

np.int64(5413)

In [850]:
rfd = encounter_series.time.iloc[-10]

In [851]:
rfd

Timestamp('2023-02-13 01:00:00')

In [852]:
outcome = final_cohort[final_cohort.encounterId==EID].outcome.iloc[0]

In [853]:
outcome

np.int64(1)

In [854]:
sample_times = []
t = rfd.round('h')
while t >= (final_cohort[final_cohort.encounterId==EID].admitStoreDateTime.iloc[0] + timedelta(hours=48)):
    sample_times.append(t)
    t -= timedelta(hours=24)

In [855]:
final_cohort.columns

Index(['NHS number', 'Hospital number', 'Date of birth', 'Unit admit date',
       'Unit admit time', 'Admission type',
       'Readmission during this hospital stay', 'Nature of surgery',
       'Primary reason for admission to Unit',
       'Declared clinically ready for discharge',
       'Clinically ready for discharge (date)',
       'Clinically ready for discharge (time)', 'Discharged/died on (date)',
       'Discharged/died at (time)', 'Unit stay (days)', 'Unit outcome',
       'Delay from request to discharge (mins)', 'Discharge delay abnormal',
       'Abnormal delay caused by', 'Destination (name)',
       'Date discharged from this hospital', 'Hospital sent to (name)',
       'Outcome on discharge from this hospital',
       'Days from Unit to hospital discharge',
       'Date of ultimate hospital discharge',
       'Ultimate hospital discharge to (name)',
       'Outcome on ultimate hospital discharge',
       'Days this hosp to ultimate discharge', 'admitDay', 'icnarc_id',

In [856]:
encounter_series[encounter_series.time.isin(sample_times)]

,time,hours_since_admission,Pao2,actual bicarbonate,arterial oxygen saturation,Hemoglobin,platelet count,Sodium,potassium,creatinine,...,positive end-expiratory pressure_ffill,tidal volume_ffill,respiratory rate_ffill,peripheral oxygen saturation_ffill,Glasgow Coma Scale_ffill,Richmond Agitation-Sedation Scale_ffill,Pao2/Fio2_ffill,Urine output_ffill,Endotracheal tube_ffill,label
60,2023-02-11 01:00:00,60.0,11.20,24.1,97.1,NaN,NaN,137,5.4,NaN,...,10.0,450,17,97,3,-5,37,NaN,Yes,0
84,2023-02-12 01:00:00,84.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,156,32,98,3,-5,27,NaN,Yes,0
108,2023-02-13 01:00:00,108.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,10.0,427,22,96,3,-5,31,NaN,Yes,0


In [231]:
original_features = [
    'Glasgow Coma Scale_ffill', 'temperature_ffill', 'arterial blood pressure (mean)_ffill', 'Fio2_ffill',
    'arterial oxygen saturation_ffill', 'respiratory rate_ffill', 'Heart rate_ffill', 'actual bicarbonate_ffill',
    'Hemoglobin_ffill', 'platelet count_ffill', 'Sodium_ffill', 'potassium_ffill',
    'creatinine_ffill', 'urea_ffill', 'Bilirubin_ffill', 'positive end-expiratory pressure_ffill'
]

In [232]:
new_features = [
    c
    for c in encounter_series.columns
    if '_ffill' in c and c not in original_features and 'trach' not in c and 'delta' not in c
]

In [233]:
all_features = original_features + new_features

In [858]:
encounter_series[encounter_series.time.isin(sample_times)][
    ['time', 'hours_since_admission'] + original_features
]

,time,hours_since_admission,Glasgow Coma Scale_ffill,temperature_ffill,arterial blood pressure (mean)_ffill,Fio2_ffill,arterial oxygen saturation_ffill,respiratory rate_ffill,Heart rate_ffill,actual bicarbonate_ffill,Hemoglobin_ffill,platelet count_ffill,Sodium_ffill,potassium_ffill,creatinine_ffill,urea_ffill,Bilirubin_ffill,positive end-expiratory pressure_ffill
60,2023-02-11 01:00:00,60.0,3,36.4,82,30,97.1,17,102,24.1,108,227,137,5.4,94,9.1,9,10.0
84,2023-02-12 01:00:00,84.0,3,38.1,71,35,97.7,32,105,22.7,118,171,140,4.5,68,7.7,14,10.0
108,2023-02-13 01:00:00,108.0,3,38.4,65,35,96.4,22,133,23.8,94,189,140,4.9,174,17.9,16,10.0


#### Note: forward fill is not appropriate here (and in many cases) - e.g. this patient is no longer ventilated, but has a forward-filled PEEP. Also, things like GCS may not be recorded if the patient it well. So filling the last value masks this.

In [942]:
sum([
    (d.total_seconds() / 60**2) < 12
    for d in (final_cohort.dischargeStoreDateTime - final_cohort.admitStoreDateTime)
])

191

In [1]:
def compute_temporal_features(sample_time, encounter_series, variable_columns):
    """
    Compute temporal features for specific variable columns (by default all _ffill features).

    delta_1: change in variable value over the past 24 hours (or less if los < 24)
    delta_2: change in variable value over the first 24 hours (or less if los < 24)
    delta_3: change in variable value over los
    """
    if sample_time not in encounter_series.time.values:
        return {
            f"{var}_delta_{i}": np.nan
            for var in variable_columns
            for i in [1,2,3]
        }
        
    min_time = encounter_series.time.min()

    delta_1_time = max(
        sample_time - timedelta(hours=24),
        min_time
    )
    delta_2_time = min(
        min_time + timedelta(hours=24),
        sample_time
    )
    delta_3_time = min_time
    
    sample_time_row = encounter_series[encounter_series.time==sample_time].iloc[0]

    temporal_feature_values = {}
    for var in variable_columns:
        temporal_feature_values[f"{var}_delta_1"] = sample_time_row[var] - encounter_series[encounter_series.time==delta_1_time].iloc[0][var]
        temporal_feature_values[f"{var}_delta_2"] = encounter_series[encounter_series.time==delta_2_time].iloc[0][var] - encounter_series[encounter_series.time==min_time].iloc[0][var]
        temporal_feature_values[f"{var}_delta_3"] = sample_time_row[var] - encounter_series[encounter_series.time==delta_3_time].iloc[0][var]

    return temporal_feature_values
    

In [25]:
all_samples = None
rfd_na = 0
rfd_end = 0
D = 0
for ei, eid in enumerate(final_cohort.encounterId):

    if ei % 100 == 0:
        print(ei, eid)
    # if ei > 100:
    #     break
    encounter_series = pd.read_csv(save_to_path / f"{eid}_encounter_series.csv")
    encounter_series.time = pd.to_datetime(encounter_series.time)
    for f in all_features:
        encounter_series[f] = pd.to_numeric(encounter_series[f], errors='coerce')

    # fill any NaNs in first row, with second row value:
    srid = 1 if len(encounter_series) > 1 else 0
    encounter_series = encounter_series.apply(lambda x: x.fillna(value=encounter_series.iloc[srid,:]))
    
    rfd = final_cohort[final_cohort.encounterId==eid].rfd_datetime.iloc[0].round('h')
    admit_time = final_cohort[final_cohort.encounterId==eid].admitStoreDateTime.iloc[0].round('h')
    
    if not pd.isna(rfd):
        if rfd > encounter_series.time.max():
                rfd_end += 1
                rfd = encounter_series.time.max()
            
        sample_times = []
        t = rfd
        sample_times.append(t)
        t -= timedelta(hours=24)
        while t >= (admit_time + timedelta(hours=48)):
            sample_times.append(t)
            t -= timedelta(hours=24)

        view = encounter_series[encounter_series.time.isin(sample_times)].copy()
        view['eid'] = eid
        view['outcome'] = final_cohort[final_cohort.encounterId==eid].outcome.iloc[0]

        # add temporal features
        temp_feat = pd.DataFrame.from_records([
            compute_temporal_features(s, encounter_series, original_features)
            for s in sample_times
        ])
        for col in temp_feat.columns:
            view[col] = temp_feat[col].values
        
        delta = len(sample_times) - len(view)
        D += delta
        # if delta > 0:
            # print('not all sample times found')
            # break
        if all_samples is None:
            all_samples = view
        else:
            all_samples = pd.concat([all_samples, view])
    else:
        rfd_na+=1

# print(rfd_na)
# print(D)
# print(rfd_end)

0 1583


C:\Users\mcwilliamschr\AppData\Local\Temp\13\ipykernel_34188\2458773888.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encounter_series = encounter_series.apply(lambda x: x.fillna(value=encounter_series.iloc[srid,:]))
C:\Users\mcwilliamschr\AppData\Local\Temp\13\ipykernel_34188\2458773888.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  encounter_series = encounter_series.apply(lambda x: x.fillna(value=encounter_series.iloc[srid,:]))
C:\Users\mcwilliamschr\AppData\Local\Temp\13\ipykernel_34188\2458773888.py:18: FutureWarning: Downcasting object

TypeError: 'str' object cannot be interpreted as an integer

In [1010]:
all_samples.to_csv(save_to_path / f"all_24hr_samples_final_cohort_.csv")

In [1011]:
all_samples.groupby(['label']).count()

,Unnamed: 0,time,hours_since_admission,Pao2,actual bicarbonate,arterial oxygen saturation,Hemoglobin,platelet count,Sodium,potassium,...,creatinine_ffill_delta_3,urea_ffill_delta_1,urea_ffill_delta_2,urea_ffill_delta_3,Bilirubin_ffill_delta_1,Bilirubin_ffill_delta_2,Bilirubin_ffill_delta_3,positive end-expiratory pressure_ffill_delta_1,positive end-expiratory pressure_ffill_delta_2,positive end-expiratory pressure_ffill_delta_3
label,,,,,,,,,,,,,,,,,,,,,
0,16796,16796,16796,7843,5854,7838,1275,460,8074,8074,...,5935,16627,5946,5946,13644,5163,4829,14062,4055,4055
1,4129,4129,4129,949,831,952,197,111,1032,1030,...,1444,3598,1446,1446,2978,1215,1160,1787,650,650


In [933]:
all_samples.groupby('label').count()

,Unnamed: 0,time,hours_since_admission,Pao2,actual bicarbonate,arterial oxygen saturation,Hemoglobin,platelet count,Sodium,potassium,...,tidal volume_ffill,respiratory rate_ffill,peripheral oxygen saturation_ffill,Glasgow Coma Scale_ffill,Richmond Agitation-Sedation Scale_ffill,Pao2/Fio2_ffill,Urine output_ffill,Endotracheal tube_ffill,eid,outcome
label,,,,,,,,,,,,,,,,,,,,,
0,16796,16796,16796,7843,5854,7838,1275,460,8074,8074,...,13982,16794,16794,16792,16781,14860,16,6645,16796,16796
1,4129,4129,4129,949,831,952,197,111,1032,1030,...,1874,4128,4127,4129,4097,2447,0,903,4129,4129


In [929]:
all_samples.groupby(['outcome', 'label']).count()['eid']

outcome  label
0        0         4012
1        0        12784
         1         4129
Name: eid, dtype: int64

In [930]:
len(all_samples.eid.unique())

5213

### We now do the same feature sampling, but we take every 4 hours from 12 hours post-admission until discharge at standard time points (10:00, 14:00, 18:00, 22:00, 02:00, 06:00). 

Every hour would likely result in too many identical or-near identical smaples. And we need standardised time points with a view to deployment.

In [40]:
save_to_path = Path(f"E:/SMARTT/data/processed/{project_name}")
final_cohort = pd.read_json(
    '..\data\processed\linked_cohort_181024.json',
    convert_dates=['admitStoreDateTime', 'dischargeStoreDateTime']
)

In [30]:
pd.set_option('future.no_silent_downcasting', True)

In [54]:
start_hours_post_admission = 12
sample_freq = 4
all_samples = None
bc = 0
for ei, eid in enumerate(final_cohort.encounterId):

    if ei % 100 == 0:
        print(ei, eid)
    # if bc > 100:
    #     break
    encounter_series = pd.read_csv(save_to_path / f"{eid}_encounter_series.csv")
    encounter_series.time = pd.to_datetime(encounter_series.time)
    for f in all_features:
        encounter_series[f] = pd.to_numeric(encounter_series[f], errors='coerce')

    # fill any NaNs in first row, with second row value:
    srid = 1 if len(encounter_series) > 1 else 0
    encounter_series = encounter_series.apply(lambda x: x.fillna(value=encounter_series.iloc[srid,:]))
    
    # rfd = final_cohort[final_cohort.encounterId==eid].rfd_datetime.iloc[0].round('h')
    admit_time = final_cohort[final_cohort.encounterId==eid].admitStoreDateTime.iloc[0].round('h')
    discharge_time = final_cohort[final_cohort.encounterId==eid].dischargeStoreDateTime.iloc[0].round('h')
       
    sample_times = []
    t = admit_time + timedelta(hours=start_hours_post_admission)
    sample_times.append(t)
    t += timedelta(hours=sample_freq)
    while t <= (discharge_time):
        sample_times.append(t)
        t += timedelta(hours=sample_freq)

    view = encounter_series[encounter_series.time.isin(sample_times)].copy()
    view['eid'] = eid
    view['outcome'] = final_cohort[final_cohort.encounterId==eid].outcome.iloc[0]

    if len(view) > 0:
        bc += 1
        # add temporal features
        temp_feat = pd.DataFrame.from_records([
            compute_temporal_features(s, encounter_series, original_features)
            for s in view.time
        ])
        for col in temp_feat.columns:
            view[col] = temp_feat[col].values
        
        if all_samples is None:
            all_samples = view
        else:
            all_samples = pd.concat([all_samples, view])
    else:
        print(f"No values found at sample times: {eid}")

0 1583
No values found at sample times: 2389
100 2399
No values found at sample times: 2451
No values found at sample times: 2465
No values found at sample times: 2505
No values found at sample times: 2595
200 2711
No values found at sample times: 2741
300 3003
No values found at sample times: 3022
No values found at sample times: 3083
No values found at sample times: 3165
No values found at sample times: 3224
400 3327
No values found at sample times: 3329
No values found at sample times: 3433
No values found at sample times: 3583
No values found at sample times: 3584
500 3628
No values found at sample times: 3628
No values found at sample times: 3756
No values found at sample times: 3773
600 3946
No values found at sample times: 4161
700 4234
No values found at sample times: 4467
800 4509
No values found at sample times: 4626
No values found at sample times: 4769
No values found at sample times: 4790
900 4837
No values found at sample times: 5041
No values found at sample times: 5108


In [55]:
all_samples.to_csv(save_to_path / f"all_{sample_freq}hr_samples_from_{start_hours_post_admission}_hours_final_cohort_.csv")

In [56]:
all_samples.groupby(['label']).count()

,Unnamed: 0,time,hours_since_admission,Pao2,actual bicarbonate,arterial oxygen saturation,Hemoglobin,platelet count,Sodium,potassium,...,creatinine_ffill_delta_3,urea_ffill_delta_1,urea_ffill_delta_2,urea_ffill_delta_3,Bilirubin_ffill_delta_1,Bilirubin_ffill_delta_2,Bilirubin_ffill_delta_3,positive end-expiratory pressure_ffill_delta_1,positive end-expiratory pressure_ffill_delta_2,positive end-expiratory pressure_ffill_delta_3
label,,,,,,,,,,,,,,,,,,,,,
0,249941,249941,249941,127578,94779,127385,33494,22701,137405,137323,...,87482,228733,87619,87619,192880,75245,71341,185595,61677,61677
1,26826,26826,26826,2706,2753,2705,1238,1093,3607,3599,...,9120,25696,9115,9115,21471,7753,7302,12818,4055,4055
